In [1]:

import gradio as gr
import ollama

# ----------------------------------
# Balanced Diet Recommendation Chatbot
# ----------------------------------

MODEL_NAME = "qwen2.5:0.5b-instruct"

SYSTEM_PROMPT = """
You are a balanced diet information assistant.

Your purpose is to provide general, safe dietary information.

Include these food categories when relevant:
1. Millets: Ragi, foxtail millet, little millet, and barnyard millet.
2. Nuts and seeds: Almonds, walnuts, chia seeds, and flaxseeds.
3. Fruits: Apples, guavas, oranges, papayas, and other suitable fruits.
4. Vegetables: Leafy greens, carrots, beans, and other vegetables.
5. Protein sources: Lentils, beans, chickpeas, eggs, fish, and other
   suitable protein sources based on the user's preferences.
6. Whole grains and healthy fats.

Support general dietary questions concerning:
- Diabetes
- High blood pressure
- Thyroid conditions
- Weight management
- General health
- Vegetarian and non-vegetarian diets

Response guidelines:
- Use simple English.
- Suggest balanced meals rather than individual miracle foods.
- Include breakfast, lunch, dinner, and snack ideas when appropriate.
- Explain that portion sizes and dietary requirements vary by person.
- Do not promise to cure or treat any medical condition.
- Do not recommend stopping prescribed medicines.
- For medical conditions, pregnancy, food allergies, or special diets,
  recommend consulting a qualified healthcare professional.
- For thyroid conditions, explain that diet depends on the specific
  condition and prescribed treatment.
- Avoid claiming that any food is completely cholesterol-free or
  automatically safe for every person.
- Ask follow-up questions when important information is missing.
"""

def get_diet_recommendation(
    health_condition,
    food_preference,
    age_group,
    goal,
    question
):
    if not question.strip():
        return "Please enter your diet-related question."

    user_prompt = f"""
    User information:
    Health condition: {health_condition}
    Food preference: {food_preference}
    Age group: {age_group}
    Dietary goal: {goal}

    User question:
    {question}

    Provide a practical balanced diet suggestion.

    Include suitable millets, vegetables, fruits, nuts, and protein
    sources where appropriate.

    Mention relevant food precautions and recommend professional
    medical advice when necessary.
    """

    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

        return response["message"]["content"]

    except Exception as error:
        return (
            "Unable to connect to the Ollama model.\n\n"
            "Please check the following:\n"
            "1. Ollama is installed and running.\n"
            "2. The model is downloaded.\n"
            "3. MODEL_NAME matches the name in 'ollama list'.\n\n"
            f"Technical details: {error}"
        )


# ----------------------------------
# Gradio User Interface
# ----------------------------------

with gr.Blocks(title="Balanced Diet AI Assistant") as demo:

    gr.Markdown(
        """
        # 🥗 Balanced Diet AI Assistant  (only for learning purpose)

        ### Healthy food ideas using Millets, Organic Foods,
        Nuts, Fruits, Vegetables and Protein-rich foods.
        
        Enter your preferences and ask your diet-related question.
        """
    )

    with gr.Row():
        health_condition = gr.Dropdown(
            choices=[
                "No known health condition",
                "Diabetes",
                "High blood pressure",
                "Thyroid condition",
                "Weight management",
                "Other"
            ],
            value="No known health condition",
            label="Health Condition"
        )

        food_preference = gr.Dropdown(
            choices=[
                "Vegetarian",
                "Vegan",
                "Non-vegetarian",
                "Eggitarian"
            ],
            value="Vegetarian",
            label="Food Preference"
        )

    with gr.Row():
        age_group = gr.Dropdown(
            choices=[
                "Child",
                "Teenager",
                "Adult",
                "Older adult"
            ],
            value="Adult",
            label="Age Group"
        )

        goal = gr.Dropdown(
            choices=[
                "General health",
                "Weight management",
                "Balanced nutrition",
                "Blood sugar awareness",
                "Heart-healthy eating"
            ],
            value="Balanced nutrition",
            label="Dietary Goal"
        )

    question = gr.Textbox(
        label="Ask Your Question",
        placeholder=(
            "Example: Suggest a balanced breakfast using "
            "ragi, nuts, and fruit."
        ),
        lines=3
    )

    submit_button = gr.Button(
        "Get Diet Recommendation"
    )

    output = gr.Textbox(
        label="AI Diet Recommendation",
        lines=15
    )

    submit_button.click(
        fn=get_diet_recommendation,
        inputs=[
            health_condition,
            food_preference,
            age_group,
            goal,
            question
        ],
        outputs=output
    )

    gr.Markdown(
        """
        **Disclaimer:** This chatbot provides general food information,
        not medical diagnosis or treatment. People with medical
        conditions should consult a qualified healthcare professional
        for personalized dietary advice.
        """
    )


# ----------------------------------
# Run the Application
# ----------------------------------

if __name__ == "__main__":
    demo.launch()



* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
